# WhisperBench — Audio Dataset Quality Pipeline
Ingests raw audio, transcribes via Whisper, scores quality metrics, filters low-quality samples, exports clean Parquet dataset.

**Runtime:** ~60-90 min on T4 for LibriSpeech test-clean (~2600 files, ~5hrs audio)

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q openai-whisper pyannote.audio pandas pyarrow soundfile librosa tqdm
!apt-get install -q ffmpeg

In [ ]:
# ── Cell 2: Download LibriSpeech test-clean ────────────────────────────────────
import os, tarfile, urllib.request

URL  = "https://www.openslr.org/resources/12/test-clean.tar.gz"
DEST = "test-clean.tar.gz"

if not os.path.exists("LibriSpeech"):
    print("Downloading LibriSpeech test-clean (~346 MB)...")
    urllib.request.urlretrieve(URL, DEST)
    print("Extracting...")
    with tarfile.open(DEST) as t:
        t.extractall()
    os.remove(DEST)
    print("Done.")
else:
    print("Already downloaded.")

In [ ]:
# ── Cell 3: Collect all .flac files ───────────────────────────────────────────
import glob

audio_files = sorted(glob.glob("LibriSpeech/**/*.flac", recursive=True))
print(f"Found {len(audio_files)} audio files")

In [ ]:
# ── Cell 4: Load Whisper model ─────────────────────────────────────────────────
import whisper

# 'base' is fast and good enough for quality scoring; swap to 'small' for better accuracy
model = whisper.load_model("base")
print("Whisper 'base' loaded")

In [ ]:
# ── Cell 5: Quality scoring functions ─────────────────────────────────────────
import numpy as np
import librosa

def compute_snr(path):
    """Signal-to-noise ratio estimate via RMS of signal vs noise floor."""
    y, sr = librosa.load(path, sr=None, mono=True)
    # Estimate noise from quietest 10% of frames
    frame_rms = librosa.feature.rms(y=y)[0]
    noise_floor = np.percentile(frame_rms, 10)
    signal_rms  = np.mean(frame_rms)
    if noise_floor < 1e-10:
        return 99.0
    snr = 20 * np.log10(signal_rms / noise_floor)
    return round(float(snr), 2)

def get_duration(path):
    """Duration in seconds."""
    y, sr = librosa.load(path, sr=None, mono=True)
    return round(len(y) / sr, 3)

def transcribe(path):
    """Returns transcript text and avg log-prob (confidence proxy)."""
    result = model.transcribe(path, language="en", fp16=True)
    text   = result["text"].strip()
    # avg_logprob per segment (Whisper internal confidence)
    segs   = result.get("segments", [])
    if segs:
        avg_logprob = float(np.mean([s["avg_logprob"] for s in segs]))
    else:
        avg_logprob = -1.0
    return text, round(avg_logprob, 4)

print("Scoring functions ready")

In [ ]:
# ── Cell 6: Run pipeline ───────────────────────────────────────────────────────
# NOTE: Full dataset (~2600 files) takes ~60-90 min on T4.
# Set LIMIT to e.g. 200 for a quick test run first.
import time
from tqdm import tqdm

LIMIT = None  # Set to 200 for quick test, None for full run

files_to_process = audio_files[:LIMIT] if LIMIT else audio_files

records = []
errors  = []
t0      = time.time()

for path in tqdm(files_to_process, desc="Processing"):
    try:
        duration   = get_duration(path)
        snr        = compute_snr(path)
        transcript, confidence = transcribe(path)
        records.append({
            "file":       os.path.basename(path),
            "path":       path,
            "duration_s": duration,
            "snr_db":     snr,
            "transcript": transcript,
            "confidence": confidence,
        })
    except Exception as e:
        errors.append({"file": path, "error": str(e)})

elapsed = time.time() - t0
print(f"\nProcessed {len(records)} files in {elapsed/60:.1f} min")
print(f"Errors: {len(errors)}")
print(f"Throughput: {len(records)/(elapsed/60):.1f} files/min")

In [ ]:
# ── Cell 7: Apply quality filters & export ─────────────────────────────────────
import pandas as pd

df = pd.DataFrame(records)

# ── Quality thresholds (tune these) ──
MIN_DURATION_S  = 1.0    # drop clips shorter than 1 second
MAX_DURATION_S  = 30.0   # drop unusually long clips
MIN_SNR_DB      = 10.0   # drop low-SNR (noisy) clips
MIN_CONFIDENCE  = -0.8   # drop low-confidence transcripts (Whisper log-prob)
MIN_WORDS       = 2      # drop near-empty transcripts

df["word_count"] = df["transcript"].apply(lambda t: len(t.split()))

mask_duration   = df["duration_s"].between(MIN_DURATION_S, MAX_DURATION_S)
mask_snr        = df["snr_db"]    >= MIN_SNR_DB
mask_confidence = df["confidence"] >= MIN_CONFIDENCE
mask_words      = df["word_count"] >= MIN_WORDS

df["passed"] = mask_duration & mask_snr & mask_confidence & mask_words

df_clean    = df[df["passed"]].drop(columns=["passed"])
df_rejected = df[~df["passed"]].drop(columns=["passed"])

total_hours      = df["duration_s"].sum() / 3600
clean_hours      = df_clean["duration_s"].sum() / 3600
filter_rate      = len(df_rejected) / len(df) * 100

print(f"Total files:       {len(df)}")
print(f"Total audio:       {total_hours:.2f} hours")
print(f"Passed filter:     {len(df_clean)} files ({clean_hours:.2f} hrs)")
print(f"Rejected:          {len(df_rejected)} files ({filter_rate:.1f}%)")
print(f"\nFilter breakdown:")
print(f"  Failed duration:    {(~mask_duration).sum()}")
print(f"  Failed SNR:         {(~mask_snr).sum()}")
print(f"  Failed confidence:  {(~mask_confidence).sum()}")
print(f"  Failed word count:  {(~mask_words).sum()}")

# Export
df_clean.to_parquet("whisperbench_clean.parquet", index=False)
df_rejected.to_parquet("whisperbench_rejected.parquet", index=False)
df.to_parquet("whisperbench_all.parquet", index=False)
print("\nExported: whisperbench_clean.parquet, whisperbench_rejected.parquet, whisperbench_all.parquet")

In [ ]:
# ── Cell 8: Summary stats (screenshot this for your resume metrics) ────────────
print("=== WHISPERBENCH SUMMARY ===")
print(f"Files processed:   {len(df)}")
print(f"Total audio:       {total_hours:.2f} hrs")
print(f"Clean dataset:     {len(df_clean)} files / {clean_hours:.2f} hrs")
print(f"Filter rate:       {filter_rate:.1f}%")
print(f"Avg SNR (clean):   {df_clean['snr_db'].mean():.1f} dB")
print(f"Avg confidence:    {df_clean['confidence'].mean():.3f}")
print(f"Throughput:        {len(records)/(elapsed/60):.1f} files/min")
print()
print("--- Sample clean records ---")
print(df_clean[["file","duration_s","snr_db","confidence","transcript"]].head(5).to_string(index=False))